In [15]:
import pandas as pd
df=pd.read_csv("./raw_dataset.csv")
df.head(2)

,Timestamp,Gender,Level of Study,Age,Field of Study,CGPA,How often do you take exams?,Which system allows you to better demonstrate your understanding?,Which exam type causes you more stress?,If you could design the perfect examination system for your field what features would it include?,MCQ exams are fair because grading is objective.,Written exams are fair despite potential subjectivity.,MCQ exams accurately reflect my knowledge.,Guessing can improve scores in MCQ exams.,Written exams encourage deeper understanding.,MCQ exams are easier to prepare for than written exams.,Written exams better measure understanding.,I often run out of time during written exams.,I experience higher stress during written exams than MCQ exams.
0,4/7/2026 14:55:30,Male,Undergraduate,18-22,Mechatronics & Robotics Engineering,3,Very often,MCQ exams,Written exams,Projects,5,3,5,5,5,5,4,3,4
1,4/7/2026 15:26:33,Male,Undergraduate,18-22,Data science,3.3,Very often,Written exams,Both equally,Balanced system which contains both MCQ and wr...,3,4,2,5,5,3,5,3,1


## Fix Typo


In [16]:
df['How often do you take exams?'].unique()

array(['Very often', 'Sometimes', 'Rearely'], dtype=object)

In [17]:
df['How often do you take exams?']=df['How often do you take exams?'].replace("Rearely",'Rarely')


## Handle open-ended fields
### Group to categories

In [ ]:
# Field of study
import numpy as np 

def categorize_study(val):
  val=str(val).lower().strip()

  # 1. Data Science & AI
  if any(keyword in val for keyword in['data sci','data sin','data scin','data sci','fcds','artificial intelligence','data','حاسبات و علوك البيانات']):
    return 'Data Science & AI'
  # 2. Computer Science
  elif any(keyword in val for keyword in ['computer science','cs','programming','it','computers']):
    return 'Computer Science & IT'
  # 3. Engineering
  elif any (keyword in val for keyword in ['engineer','cce','Mechatronics','Robotics','Mechatronics','electrical','civil']):
    return 'Engineering'
  # 4. Business
  elif any (keyword in val for keyword in['business','accounting','economics','analytics','administration']):
    return 'Business & Economics'
  # 5. Medical
  elif any(keyword in val for keyword in ['medicine', 'pharmacy', 'clinical', 'pharmacist']):
    return 'Medical & Health'
  # 6. Law & Humanities
  elif any(keyword in val for keyword in ['law', 'حقوق', 'lit', 'psychology', 'educat', 'lang']):
    return 'Humanities & Law'
  # 7. high school student
  elif '-'in val:
    return "High School Student"
  
  return 'Other'


# Apply the change
df['Field_Category']=df['Field of Study'].apply(categorize_study)
print(df['Field_Category'].value_counts())

Field_Category
Data Science & AI        92
Computer Science & IT    42
Other                    14
Engineering              12
High School Student       9
Humanities & Law          8
Business & Economics      6
Medical & Health          5
Name: count, dtype: int64


In [24]:
# Perfect Examination system 

def categorize_exam_preference(df):
    text = df['If you could design the perfect examination system for your field what features would it include?'].str.lower().fillna('')

    conditions = [
        # 1. Practical / Project based
        text.str.contains('project|practical|coding|lab|job market|task', na=False),
        
        # 2. Mixed 
        (text.str.contains('mcq', na=False) & text.str.contains('written|essay|short answer|مقال', na=False)) | 
        text.str.contains('mix|both|balance|combination|hybrid|equally', na=False),
        
        # 3. MCQ 
        text.str.contains('mcq|choice|امسكيو|اختيار| M| not written', na=False),
        
        # 4. Written 
        text.str.contains('written|essay|short answer|مقال|بكتب |not mcq | W', na=False)
    ]

    # Define the category names
    choices = [
        'Practical/Project-Based',
        'Mixed ',
        'MCQ ',
        'Written '
    ]

    # Apply categorie
    df['Preferred_System_Type'] = np.select(conditions, choices, default='Other')
    
    return df

df = categorize_exam_preference(df)

print(df['Preferred_System_Type'].value_counts())

Preferred_System_Type
Other                      59
MCQ                        45
Mixed                      45
Practical/Project-Based    23
Written                    16
Name: count, dtype: int64


## Encoding 

In [ ]:
# Rename columns to match my encoding 
rename_map={
  'Level of Study':"Level",
  'How often do you take exams?':"Exam Frequency",
  'Which exam type causes you more stress?':"Stress Exam Type",
  'Which system allows you to better demonstrate your understanding?':"Better System",
}
df=df.rename(columns=rename_map)

In [ ]:
# Encoding 
encoding_maps = {
    'Gender': {'Male': 1, 'Female': 0},
    'Level': {'High School': 0, 'Undergraduate': 1, 'Postgraduate': 2},
    'Age': {'Under 18': 0, '18-22': 1, '23-28': 2},
    'Exam Frequency': {'Rarely': 0, 'Sometimes': 1, 'Very often': 2},
    'Stress Exam Type': {'MCQ exams': 0, 'Written exams': 1, 'Both equally': 2},
    'Better System': {'MCQ exams': 0, 'Written exams': 1, 'Both equally': 2}
}

# Step 3: Apply the encoding
for column, mapping in encoding_maps.items():
    if column in df.columns:
        df[column] = df[column].map(mapping)

print("Columns successfully renamed and encoded!")
print(df.head())

Columns successfully renamed and encoded!
           Timestamp  Gender  Level  Age  \
0  4/7/2026 14:55:30       1      1    1   
1  4/7/2026 15:26:33       1      1    1   
2  4/7/2026 17:07:47       0      0    0   
3  4/7/2026 18:28:33       0      1    1   
4  4/7/2026 19:11:23       1      1    1   

                         Field of Study CGPA  Exam Frequency  Better System  \
0  Mechatronics & Robotics Engineering     3               2              0   
1                         Data science   3.3               2              1   
2                                     -    -               2              0   
3         دراسات قانونيه ومعاملات دوليه  2.7               2              2   
4                         Data science   3.2               1              0   

   Stress Exam Type  \
0                 1   
1                 2   
2                 0   
3                 1   
4                 1   

  If you could design the perfect examination system for your field what featur

In [34]:
# Rename Likert Questions
likert_rename = {
    'MCQ exams are fair because grading is objective.': 'Likert_MCQ_Objectivity',
    'Written exams are fair despite potential subjectivity. ': 'Likert_Written_Fairness',
    'MCQ exams accurately reflect my knowledge.': 'Likert_MCQ_Accuracy',
    'Guessing can improve scores in MCQ exams.   ': 'Likert_MCQ_Guessing',
    'Written exams encourage deeper understanding.   ': 'Likert_Written_Depth',
    'MCQ exams are easier to prepare for than written exams.   ': 'Likert_MCQ_Prep_Ease',
    'Written exams better measure understanding.   ': 'Likert_Written_Measure',
    'I often run out of time during written exams.  ': 'Likert_Written_Time_Stress',
    'I experience higher stress during written exams than MCQ exams.  ': 'Likert_Stress_Comparison'
}
df = df.rename(columns=likert_rename)

In [35]:
df.columns

Index(['Timestamp', 'Gender', 'Level', 'Age', 'Field of Study', 'CGPA',
       'Exam Frequency', 'Better System', 'Stress Exam Type',
       'If you could design the perfect examination system for your field what features would it include?',
       'Likert_MCQ_Objectivity', 'Likert_Written_Fairness',
       'Likert_MCQ_Accuracy', 'Likert_MCQ_Guessing', 'Likert_Written_Depth',
       'Likert_MCQ_Prep_Ease', 'Likert_Written_Measure',
       'Likert_Written_Time_Stress', 'Likert_Stress_Comparison',
       'Field_Category', 'Preferred_System_Type'],
      dtype='object')

In [36]:
# List of columns to remove
cols_to_drop = [
    'Timestamp', 
    'Field of Study', 
    'If you could design the perfect examination system for your field what features would it include?'
]

# Drop the columns
df = df.drop(columns=cols_to_drop)

print(df.columns)

Index(['Gender', 'Level', 'Age', 'CGPA', 'Exam Frequency', 'Better System',
       'Stress Exam Type', 'Likert_MCQ_Objectivity', 'Likert_Written_Fairness',
       'Likert_MCQ_Accuracy', 'Likert_MCQ_Guessing', 'Likert_Written_Depth',
       'Likert_MCQ_Prep_Ease', 'Likert_Written_Measure',
       'Likert_Written_Time_Stress', 'Likert_Stress_Comparison',
       'Field_Category', 'Preferred_System_Type'],
      dtype='object')


## Save to new CSV File

In [38]:
df.to_csv('cleaned_dataset.csv',index=False)
